# Analyze Archive Edge-Flip Causality

For each of the best archived K43 colorings, find its best bare exact-greedy single-edge flip and measure the complete monochromatic-K5 causal footprint. This notebook collects information only; it does not mutate or archive any coloring.

In [10]:
# Imports and Configuration

from pathlib import Path

import numpy as np

from ramsey import (
    RGraph,
    RProblem,
    RSQLiteArchive,
    RSearchState,
)
from ramsey.REdgeFlipCausalAnalysis import (
    analyze_edge_flip_causality,
)
from ramsey.RAction import analyze_actions

N_VERTICES = 43
ANALYSIS_LIMIT = 1

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [11]:
# Load the Best Archive States

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,

    )
)

archive = RSQLiteArchive(DATABASE_PATH)
records = archive.colorings_in_score_range(
    minimum_score=350,
    maximum_score=399,
    limit=ANALYSIS_LIMIT,
    graph=graph,
)

print("Database:", DATABASE_PATH.resolve())
print("Archive best:", archive.best_score(graph))
print("States selected:", len(records))
print("Scores:", [record.score for record in records])
# Analyze the Best Exact-Greedy Flip in Every State

analyses = []

for record in records:
    archived = archive.load_coloring(
        record.coloring_id,
        graph,
    )
    state = RSearchState(
        archived.coloring
    )

    action_analysis = analyze_actions(state)
    best_reward = int(
        action_analysis.immediate_rewards.max()
    )
    best_edges = np.flatnonzero(
        action_analysis.immediate_rewards
        == best_reward
    )
    edge = int(best_edges[0])

    causal = analyze_edge_flip_causality(
        state,
        edge,
    )

    destroyed_red = sum(
        change.destroyed and change.color == 0
        for change in causal.clique_changes
    )
    destroyed_blue = sum(
        change.destroyed and change.color == 1
        for change in causal.clique_changes
    )
    created_red = sum(
        change.created and change.color == 0
        for change in causal.clique_changes
    )
    created_blue = sum(
        change.created and change.color == 1
        for change in causal.clique_changes
    )

    analyses.append({
        "record": record,
        "causal": causal,
        "tied_best_edges": len(best_edges),
        "destroyed_red": destroyed_red,
        "destroyed_blue": destroyed_blue,
        "created_red": created_red,
        "created_blue": created_blue,
        "changed_vertices": len(causal.changed_vertices),
        "structure_edges": len(causal.changed_structure_edges),
        "changed_future_rewards": int(
            np.count_nonzero(
                causal.greedy_reward_delta
            )
        ),
        "max_vertex_hits": int(
            causal.vertex_event_counts.max()
        ),
        "max_edge_hits": int(
            causal.edge_event_counts.max()
        ),
    })

print(
    " ID    score  edge  endpoints  color  reward  "
    "destroy R/B  create R/B  verts  edges  future  ties"
)
print("-" * 105)

for item in analyses:
    record = item["record"]
    causal = item["causal"]
    old_color = "R" if causal.old_color == 0 else "B"
    new_color = "B" if causal.new_color == 1 else "R"

    print(
        f"{record.coloring_id:5d}  "
        f"{record.score:5d}  "
        f"{causal.edge:4d}  "
        f"{str(causal.endpoints):>9s}  "
        f"{old_color}->{new_color}   "
        f"{causal.exact_reward:+5d}     "
        f"{item['destroyed_red']:3d}/{item['destroyed_blue']:<3d}    "
        f"{item['created_red']:3d}/{item['created_blue']:<3d}   "
        f"{item['changed_vertices']:3d}   "
        f"{item['structure_edges']:3d}    "
        f"{item['changed_future_rewards']:3d}   "
        f"{item['tied_best_edges']:3d}"
    )
# Aggregate Causal Statistics

rewards = np.asarray([
    item["causal"].exact_reward
    for item in analyses
])
changed_vertices = np.asarray([
    item["changed_vertices"]
    for item in analyses
])
structure_edges = np.asarray([
    item["structure_edges"]
    for item in analyses
])
future_changes = np.asarray([
    item["changed_future_rewards"]
    for item in analyses
])

print("States:", len(analyses))
print("Positive best greedy reward:", int(np.count_nonzero(rewards > 0)))
print("Zero best greedy reward:", int(np.count_nonzero(rewards == 0)))
print("Negative best greedy reward:", int(np.count_nonzero(rewards < 0)))
print("Mean best greedy reward:", f"{rewards.mean():+.2f}")
print("Mean changed vertices:", f"{changed_vertices.mean():.2f} / 43")
print("Mean causal-structure edges:", f"{structure_edges.mean():.2f} / 903")
print("Mean changed future rewards:", f"{future_changes.mean():.2f} / 903")
print("Maximum changed future rewards:", int(future_changes.max()))
# Inspect the Lowest-Score Archive State Vertex by Vertex

item = analyses[0]
record = item["record"]
causal = item["causal"]
before = causal.participation_before.vertices
delta = causal.vertex_participation_delta
after = causal.participation_after.vertices
hits = causal.vertex_event_counts

print("Archive ID:", record.coloring_id)
print("Score:", record.score)
print("Greedy edge:", causal.edge, causal.endpoints)
print("Reward:", causal.exact_reward)
print()
print("vertex   before(R,B)   delta(R,B)   after(R,B)   event hits")
print("-" * 67)

for vertex in causal.changed_vertices:
    vertex = int(vertex)
    print(
        f"{vertex:6d}   "
        f"({before[vertex, 0]:3d},{before[vertex, 1]:3d})   "
        f"({delta[vertex, 0]:+3d},{delta[vertex, 1]:+3d})   "
        f"({after[vertex, 0]:3d},{after[vertex, 1]:3d})   "
        f"{hits[vertex]:5d}"
    )

Database: C:\code\RamseyNumber\data\ramsey_colorings.sqlite3
Archive best: 129
States selected: 1
Scores: [350]
 ID    score  edge  endpoints  color  reward  destroy R/B  create R/B  verts  edges  future  ties
---------------------------------------------------------------------------------------------------------
 1051    350   767   (26, 27)  B->R      +5       0/7        2/0      14    44    129     1
States: 1
Positive best greedy reward: 1
Zero best greedy reward: 0
Negative best greedy reward: 0
Mean best greedy reward: +5.00
Mean changed vertices: 14.00 / 43
Mean causal-structure edges: 44.00 / 903
Mean changed future rewards: 129.00 / 903
Maximum changed future rewards: 129
Archive ID: 1051
Score: 350
Greedy edge: 767 (26, 27)
Reward: 5

vertex   before(R,B)   delta(R,B)   after(R,B)   event hits
-------------------------------------------------------------------
     1   ( 23, 17)   ( +2, +0)   ( 25, 17)       2
     5   ( 26, 19)   ( +0, -3)   ( 26, 16)       3
     6   ( 26,